# Áreas Protegidas

> **Contexto de dominio:** [`docs/fuentes/areas_protegidas.md`](../../docs/fuentes/areas_protegidas.md)  
> **Bloque:** A | **Línea:** `areas_protegidas`  
> **Variable principal:** `cobertura_ha` (ha)  
> **Modelos sugeridos:** RANDOM_FOREST, XGBOOST  
> Flujo: `Plan.md` sección 3 — ciclo estadístico completo.

**Antes de comenzar:** Leer `docs/fuentes/areas_protegidas.md` para entender:
- Variables ambientales clave y sus rangos físicos
- Normativa colombiana aplicable (umbrales normativos)
- Fuentes de datos oficiales y frecuencia de actualización
- Preguntas analíticas típicas de esta línea

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "areas_protegidas"
VARIABLE = "cobertura_ha"
UNIDAD = "ha"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `areas_protegidas` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "areas_protegidas.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos
> Colocar el archivo en `data/raw/` y ajustar la ruta.  
> Ver sección **Datos y fuentes** de `docs/fuentes/areas_protegidas.md` para las fuentes oficiales.

In [ ]:
# df = load_csv("data/raw/areas_protegidas.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "cobertura_ha": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

## 2. Validación y EDA
> `validate()` usa rangos físicos específicos para `areas_protegidas` desde `config.py`.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_areas_protegidas.html",
       title="EDA — Áreas Protegidas", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

In [ ]:
plot_series(df, "fecha", "cobertura_ha", title="Áreas Protegidas — cobertura_ha (ha)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "cobertura_ha", period="month")
plt.show()

## 4. Estadística descriptiva

In [ ]:
summarize(df)

## 5. Inferencial
> ADR-004: Estacionariedad obligatoria pre-ARIMA (ADF + KPSS juntos).

In [ ]:
ts = df.set_index("fecha")["cobertura_ha"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

<!-- ENRICHMENT: areas_protegidas -->

## 5c. Random Forest — Prediccion de deforestacion en areas protegidas

El **Random Forest** es el modelo de referencia del IDEAM/SMByC para predecir tasas de deforestacion. Ventajas frente a modelos lineales:

- Captura interacciones no lineales entre distancia a vias, pendiente, densidad demografica
- Robusto ante outliers y variables con diferentes escalas
- Produce importancia de variables (feature importance) para priorizar monitoreo

Variables predictoras tipicas del SMByC:
| Variable | Tipo | Fuente |
|---|---|---|
| Distancia a vias | Continua | IGAC/INVIAS |
| Pendiente | Continua | DEM NASA/IGAC |
| Densidad poblacional | Continua | DANE |
| Cobertura previa | Categorica | Corine Land Cover |
| Precipitacion media | Continua | IDEAM |
| Presencia mineria | Binaria | ANM/ANLA |

> **Matriz de confusion ajustada por area:** el SMByC usa exactitudes del usuario y productor ponderadas por el area real de cada clase (no por pixeles), siguiendo Olofsson et al. (2014) — estandar IPCC Tier 3.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix, classification_report

np.random.seed(42)
N = 500  # puntos de muestreo en el area protegida

# Features: predictores de deforestacion
dist_vias_km  = np.random.exponential(15, N)           # km al vial mas cercano
pendiente_deg = np.random.gamma(2, 8, N).clip(0, 60)   # grados
dens_pob      = np.random.lognormal(2, 1, N)           # hab/km2
precipitacion = np.random.normal(2000, 400, N)          # mm/ano
mineria       = (np.random.random(N) < 0.15).astype(int)  # presencia mineria

X_rf = np.column_stack([dist_vias_km, pendiente_deg, dens_pob, precipitacion, mineria])
feature_names = ['dist_vias_km','pendiente_deg','dens_pob','precipitacion','mineria']

# Probabilidad de deforestacion (proceso generativo)
prob_def = 1 / (1 + np.exp(
    2 - 0.15*dist_vias_km + 0.02*dens_pob - 0.01*pendiente_deg + 1.5*mineria))
y_rf = (prob_def > 0.5).astype(int)  # 1=deforestado, 0=estable

# Entrenar RandomForest
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
cv_scores = cross_val_score(rf, X_rf, y_rf, cv=5, scoring='f1')
rf.fit(X_rf, y_rf)
y_pred_rf = rf.predict(X_rf)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Panel A: Importancia de variables
importances = rf.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
ax1.barh([feature_names[i] for i in sorted_idx], importances[sorted_idx],
         color=['#e74c3c' if i == sorted_idx[0] else '#3498db' for i in range(len(feature_names))],
         alpha=0.85)
ax1.set_title('Random Forest — Importancia de variables\n(prediccion deforestacion SMByC)', fontweight='bold')
ax1.set_xlabel('Importancia (Gini)'); ax1.grid(axis='x', alpha=0.3)

# Panel B: Matriz de confusion
cm = confusion_matrix(y_rf, y_pred_rf)
cm_norm = cm / cm.sum(axis=1, keepdims=True)  # normalizada por clase (exactitud productor)
im = ax2.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax2, label='Proporcion')
for i in range(2):
    for j in range(2):
        ax2.text(j, i, f'{cm[i,j]}\n({cm_norm[i,j]:.2f})',
                 ha='center', va='center', fontsize=10)
ax2.set_xticks([0,1]); ax2.set_yticks([0,1])
ax2.set_xticklabels(['Pred: Estable','Pred: Deforest.'])
ax2.set_yticklabels(['Real: Estable','Real: Deforest.'])
ax2.set_title('Matriz de confusion RF — Exactitud productor/usuario\n(Olofsson 2014 requiere ajuste por area)', fontweight='bold')

plt.tight_layout(); plt.show()
print(f'CV F1-score (5-fold): {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}')
print(f'Variable mas importante: {feature_names[importances.argmax()]} ({importances.max():.3f})')
print(classification_report(y_rf, y_pred_rf, target_names=['Estable','Deforestado']))

## 5b. Análisis de excedencias normativas
> Compara `cobertura_ha` contra las normas colombianas relevantes.

In [ ]:
rep = exceedance_report(df["cobertura_ha"], variable="cobertura_ha")
if rep.empty:
    print("Sin normas colombianas registradas para 'cobertura_ha'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

In [ ]:
df_clean = impute(df.copy(), cols=["cobertura_ha"], method="linear")
print(f"Faltantes antes: {df["cobertura_ha"].isna().sum()} | "
      f"después: {df_clean["cobertura_ha"].isna().sum()}")

## 7. Modelos predictivos

In [ ]:
ts = df_clean.set_index("fecha")["cobertura_ha"]

models = {
    "RandomForest": get_model("random_forest", lags=[1,2,3,6,12]),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 7b. Guardrails y supuestos metodológicos
<!-- GUARDRAILS: areas_protegidas -->

> **Antes de publicar resultados**, verificar que se cumplen los supuestos clave del flujo. Esta sección lista los más comunes y los específicos de la línea.

### Supuestos comunes (todas las líneas)

- **Estacionariedad (ADR-004):** ADF + KPSS deben coincidir antes de aplicar ARIMA. Si discrepan, diferenciar conservadoramente o usar modelos no-ARIMA (Prophet, ML).
- **Outliers (ADR-002):** los picos ambientales son señal real (eventos, episodios). No aplicar clipping automático — sólo `preprocessing/outliers.py` opt-in y documentado.
- **Métrica primaria (ADR-003):** RMSLE NO en variables que pueden ser negativas o cero. Usar MAE + sMAPE (o NSE / KGE en hidrología) como default.
- **Tamaño muestral mínimo:** ARIMA requiere ≥ 36 observaciones; STL anual con datos diarios, ≥ 2 ciclos completos.
- **Residuos (post-fit):** verificar normalidad (Jarque-Bera) e independencia (Ljung-Box, lag = 12). Residuos correlacionados → modelo subespecificado.
- **Walk-forward con gap (BP-1):** series con ACF ≥ 0.7 inflan R². Usar `gap ≥ horizonte` en `walk_forward()`.
- **Normas oficiales:** usar `config.NORMA_*` y `config.*_THRESHOLDS` — nunca umbrales hardcodeados en el notebook (ADR-005).

### Supuestos específicos — Áreas protegidas / GIS

- **No-serie temporal pura:** indicadores espaciales (cobertura, deforestación, fragmentación) — RF/XGBoost sobre features GIS, no ARIMA.
- **Validación cruzada espacial** (no temporal) — la autocorrelación espacial puede inflar métricas.
- **Categorías RUNAP:** SINAP, regional, municipal, sociedad civil — agregar respetando categoría.

### Antes de presentar a la autoridad ambiental

- Reportar intervalos de confianza, no sólo el punto estimado.
- Documentar el período de los datos, los gaps y el método de imputación usado.
- Registrar decisiones metodológicas no triviales en `docs/decisiones.md` (ADRs).


## 8. Conclusiones

- **Línea temática:** Áreas Protegidas (`areas_protegidas`)
- **Variable analizada:** `cobertura_ha` (ha)
- **Modelos ejecutados:** RANDOM_FOREST, XGBOOST
- Completar con hallazgos reales al trabajar con datos de producción.

### Normativa y referencias
- Ver `docs/fuentes/areas_protegidas.md` para normativa colombiana, indicadores oficiales y fuentes de datos.
- Registrar decisiones metodológicas en `docs/decisiones.md`.